# Vector Space Model

In [ ]:
documents = [
    "whoever has hate for his brother is in the darkness and walks in the darkness",
    "hello darkness my old friend",
    "returning hate for hate multiplies hate adding deeper darkness to a night already devoid of stars darkness cannot drive out darkness only light can do that"
]

## Calculating Cosine Distance in Pandas

In [ ]:
import pandas as pd
from collections import Counter

tf = pd.DataFrame(
    [Counter(doc.split()) for doc in documents],
).fillna(0)
tf

In [ ]:
import numpy as np

def length(v):
  return np.sqrt((v ** 2).sum())

def cos_dist(v, w):
  return 1 - (v * w).sum() / (length(v) * length(w))

cos_dist(tf.loc[0], tf.loc[1]), cos_dist(tf.loc[0], tf.loc[2])

## Calculating Cosine Distance in Scikit-Learn

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(token_pattern=r"\w+")
vec.fit(documents)
tf_matrix = vec.transform(documents)
tf_matrix.todense()

In [ ]:
from sklearn.metrics import pairwise_distances
pairwise_distances(tf_matrix[0, :], tf_matrix,
                   metric="cosine")

# tf-idf

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# The options ensure that the numbers match our example above.
vec = TfidfVectorizer(smooth_idf=False, norm=None)
vec.fit(documents)
tfidf_matrix = vec.transform(documents)
tfidf_matrix.todense()

In [ ]:
pairwise_distances(tfidf_matrix[0, :], tfidf_matrix,
                   metric="cosine")

# Dr. Seuss Example

What Dr. Seuss book is most similar to _One Fish, Two Fish, Red Fish, Blue Fish_?

In [1]:
seuss_dir = "http://dlsun.github.io/pods/data/drseuss/"
seuss_files = [
    "green_eggs_and_ham.txt", "cat_in_the_hat.txt",
    "fox_in_socks.txt", "how_the_grinch_stole_christmas.txt",
    "hop_on_pop.txt", "horton_hears_a_who.txt",
    "oh_the_places_youll_go.txt", "one_fish_two_fish.txt"]

In [ ]:
import requests
docs = {}
for filename in seuss_files:
    response = requests.get(seuss_dir + filename, "r")
    docs[filename] = response.text

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import pairwise_distances

# Buat TF-IDF matrix dari semua buku Dr. Seuss
vec = TfidfVectorizer()
tfidf_matrix = vec.fit_transform(docs.values())

# Hitung cosine distance antara "one_fish_two_fish.txt" dengan buku lainnya
titles = list(docs.keys())
one_fish_idx = titles.index("one_fish_two_fish.txt")

distances = pairwise_distances(tfidf_matrix[one_fish_idx, :], tfidf_matrix, metric="cosine").flatten()

# Tampilkan hasil
results = pd.DataFrame({"Book": titles, "Cosine Distance": distances})
results = results.sort_values("Cosine Distance")
print("Buku yang paling mirip dengan 'One Fish, Two Fish, Red Fish, Blue Fish':\n")
print(results.to_string(index=False))

---

# Tugas 1: Adaptasi Colab Week-4 untuk Dataset Apps Review

Mengadaptasi notebook Colab Week-4 (Sentiment Analysis IMDB) untuk dataset **Google Apps Review Indonesia** (`cleandata.csv`).

Pipeline:
1. Load & EDA dataset
2. Preprocessing & labeling sentimen biner
3. Train/Test Split
4. TF-IDF Vectorization (sklearn)
5. Klasifikasi dengan 3 model: **Linear SVM**, **Logistic Regression**, **Multinomial Naive Bayes**
6. Evaluasi & perbandingan akurasi model

## 1.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from timeit import default_timer as timer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 1.2 Load Dataset

In [ ]:
df = pd.read_csv("../Week 3/cleandata.csv")
print(f"Shape: {df.shape}")
print(f"\nKolom: {list(df.columns)}")
print(f"\nSample data:")
df.head(10)

## 1.3 Exploratory Data Analysis (EDA)

In [ ]:
# Distribusi Score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart distribusi score
score_counts = df['score'].value_counts().sort_index()
axes[0].bar(score_counts.index, score_counts.values, color=['#d32f2f', '#ff9800', '#fdd835', '#8bc34a', '#4caf50'])
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Jumlah Review')
axes[0].set_title('Distribusi Score Review')
axes[0].set_xticks([1, 2, 3, 4, 5])
for i, v in enumerate(score_counts.values):
    axes[0].text(score_counts.index[i], v + 200, str(v), ha='center', fontsize=9)

# Distribusi panjang review
df['review_length'] = df['content'].astype(str).apply(len)
axes[1].hist(df['review_length'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Panjang Review (karakter)')
axes[1].set_ylabel('Frekuensi')
axes[1].set_title('Distribusi Panjang Review')
axes[1].axvline(df['review_length'].median(), color='red', linestyle='--', label=f"Median: {df['review_length'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nStatistik panjang review:")
print(df['review_length'].describe())

## 1.4 Preprocessing & Labeling Sentimen Biner

Untuk klasifikasi sentimen biner:
- **Score ≤ 2** → `0` (Negatif)
- **Score ≥ 4** → `1` (Positif)
- **Score = 3** (Netral) → dihapus dari dataset

Kolom `text_final` digunakan karena sudah melalui preprocessing di Week 3.

In [ ]:
# Hapus score netral (3) dan buat label sentimen biner
df_sentiment = df[df['score'] != 3].copy()
df_sentiment['label'] = df_sentiment['score'].apply(lambda x: 1 if x >= 4 else 0)

# Hapus baris dengan text_final kosong/NaN
df_sentiment = df_sentiment.dropna(subset=['text_final'])
df_sentiment = df_sentiment[df_sentiment['text_final'].str.strip() != '']

print(f"Dataset setelah filter: {df_sentiment.shape[0]} baris")
print(f"\nDistribusi label sentimen:")
print(df_sentiment['label'].value_counts().rename({0: 'Negatif (0)', 1: 'Positif (1)'}))
print(f"\nRasio Positif: {df_sentiment['label'].mean():.2%}")

## 1.5 Train/Test Split

In [ ]:
X = df_sentiment['text_final'].values
y = df_sentiment['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nDistribusi label di training set:")
print(f"  Positif: {sum(y_train == 1)}, Negatif: {sum(y_train == 0)}")
print(f"Distribusi label di test set:")
print(f"  Positif: {sum(y_test == 1)}, Negatif: {sum(y_test == 0)}")

## 1.6 TF-IDF Vectorization (sklearn)

In [ ]:
# TF-IDF Vectorization dengan max 5000 fitur
tfidf_vec = TfidfVectorizer(max_features=5000)

start = timer()
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)
print(f"TF-IDF vectorization time: {timer() - start:.2f}s")

print(f"\nTF-IDF matrix shape (train): {X_train_tfidf.shape}")
print(f"TF-IDF matrix shape (test): {X_test_tfidf.shape}")
print(f"\nSample feature names (first 20): {tfidf_vec.get_feature_names_out()[:20]}")

## 1.7 Klasifikasi Sentimen

Fungsi evaluasi model dan training 3 classifier:
1. **Linear SVM** — cepat dan efektif untuk text classification
2. **Logistic Regression** — baseline yang solid
3. **Multinomial Naive Bayes** — cocok untuk fitur TF-IDF non-negatif

In [ ]:
def evaluate_model(model, X_test, y_test):
    """Evaluasi model: return predictions dan semua metrik."""
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    report = classification_report(y_test, y_pred, target_names=['Negatif', 'Positif'])
    cm = confusion_matrix(y_test, y_pred)
    return y_pred, acc, prec, rec, f1, report, cm

### Linear SVM

In [ ]:
# Training Linear SVM
start = timer()
svm_model = LinearSVC(max_iter=5000)
svm_model.fit(X_train_tfidf, y_train)
print(f"Linear SVM training time: {timer() - start:.2f}s")

# Evaluasi
start = timer()
y_pred_svm, acc_svm, prec_svm, rec_svm, f1_svm, report_svm, cm_svm = evaluate_model(svm_model, X_test_tfidf, y_test)
print(f"Linear SVM prediction time: {timer() - start:.2f}s")

print(f"\nLinear SVM Accuracy: {acc_svm:.4f}")
print(f"Linear SVM Precision: {prec_svm:.4f}")
print(f"Linear SVM Recall: {rec_svm:.4f}")
print(f"Linear SVM F1 Score: {f1_svm:.4f}")
print(f"\nClassification Report:\n{report_svm}")
print(f"Confusion Matrix:\n{cm_svm}")

### Logistic Regression

In [ ]:
# Training Logistic Regression
start = timer()
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
print(f"Logistic Regression training time: {timer() - start:.2f}s")

# Evaluasi
start = timer()
y_pred_lr, acc_lr, prec_lr, rec_lr, f1_lr, report_lr, cm_lr = evaluate_model(lr_model, X_test_tfidf, y_test)
print(f"Logistic Regression prediction time: {timer() - start:.2f}s")

print(f"\nLogistic Regression Accuracy: {acc_lr:.4f}")
print(f"Logistic Regression Precision: {prec_lr:.4f}")
print(f"Logistic Regression Recall: {rec_lr:.4f}")
print(f"Logistic Regression F1 Score: {f1_lr:.4f}")
print(f"\nClassification Report:\n{report_lr}")
print(f"Confusion Matrix:\n{cm_lr}")

### Multinomial Naive Bayes

In [ ]:
# Training Multinomial Naive Bayes
start = timer()
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
print(f"Multinomial NB training time: {timer() - start:.2f}s")

# Evaluasi
start = timer()
y_pred_nb, acc_nb, prec_nb, rec_nb, f1_nb, report_nb, cm_nb = evaluate_model(nb_model, X_test_tfidf, y_test)
print(f"Multinomial NB prediction time: {timer() - start:.2f}s")

print(f"\nMultinomial NB Accuracy: {acc_nb:.4f}")
print(f"Multinomial NB Precision: {prec_nb:.4f}")
print(f"Multinomial NB Recall: {rec_nb:.4f}")
print(f"Multinomial NB F1 Score: {f1_nb:.4f}")
print(f"\nClassification Report:\n{report_nb}")
print(f"Confusion Matrix:\n{cm_nb}")

## 1.8 Perbandingan Akurasi Model

In [ ]:
# Perbandingan akurasi semua model
model_names = ['Linear SVM', 'Logistic Regression', 'Multinomial NB']
accuracies = [acc_svm, acc_lr, acc_nb]
f1_scores = [f1_svm, f1_lr, f1_nb]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart Accuracy
colors = ['#1976d2', '#ff9800', '#4caf50']
bars1 = axes[0].bar(model_names, accuracies, color=colors)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Perbandingan Akurasi Model (TF-IDF)')
axes[0].set_ylim(0, 1.0)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
for bar, acc in zip(bars1, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, acc + 0.02, f"{acc:.4f}", ha='center', fontsize=10)

# Bar chart F1 Score
bars2 = axes[1].bar(model_names, f1_scores, color=colors)
axes[1].set_xlabel('Model')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Perbandingan F1 Score Model (TF-IDF)')
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)
for bar, f1 in zip(bars2, f1_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, f1 + 0.02, f"{f1:.4f}", ha='center', fontsize=10)

plt.tight_layout()
plt.show()

# Tabel ringkasan
print("\n=== Ringkasan Perbandingan Model ===")
summary = pd.DataFrame({
    'Model': model_names,
    'Accuracy': accuracies,
    'Precision': [prec_svm, prec_lr, prec_nb],
    'Recall': [rec_svm, rec_lr, rec_nb],
    'F1 Score': f1_scores
})
print(summary.to_string(index=False))

In [ ]:
# Confusion Matrix untuk semua model
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
cms = [cm_svm, cm_lr, cm_nb]
labels = ['Negatif', 'Positif']

for ax, cm, name in zip(axes, cms, model_names):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(f'{name}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrix - Semua Model (TF-IDF)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

# Tugas 2: Implementasi TFIDF.py (Wittline/tf-idf)

Menjalankan implementasi TF-IDF manual dari [Wittline/tf-idf](https://github.com/Wittline/tf-idf) (`TFIDF.py`) pada dataset apps review.

Class `TFIDF` memiliki pipeline:
1. `preprocessing_text()` → normalisasi corpus (lowercase, hapus karakter khusus, stopword removal)
2. `tf()` → Term Frequency matrix
3. `df(tf)` → Document Frequency
4. `idf(df)` → Inverse Document Frequency (dengan smoothing)
5. `tfidf(tf, idf)` → TF-IDF matrix (dengan L2 normalization)

In [ ]:
import sys
sys.path.insert(0, '.')
from TFIDF import TFIDF

# Gunakan sample 100 review untuk demo (full dataset terlalu besar untuk matrix dense)
sample_reviews = df_sentiment['text_final'].head(100).values

print(f"Jumlah dokumen sample: {len(sample_reviews)}")
print(f"Contoh dokumen pertama: {sample_reviews[0][:100]}...")

### 2.1 Preprocessing & Term Frequency

In [ ]:
# Inisialisasi TFIDF class
tfidf_obj = TFIDF(sample_reviews)

# Preprocessing: lowercase, hapus special chars, stopword removal
tfidf_obj.preprocessing_text()
print("Preprocessing selesai!")
print(f"Contoh setelah preprocessing: {tfidf_obj.norm_corpus[0][:100]}...")

# Hitung Term Frequency
tf_matrix = tfidf_obj.tf()
print(f"\nTF Matrix shape: {tf_matrix.shape}")
print(f"Jumlah unique terms: {tf_matrix.shape[1]}")
print(f"\nSample TF (5 dokumen pertama, 10 kolom pertama):")
tf_matrix.iloc[:5, :10]

### 2.2 Document Frequency & IDF

In [ ]:
# Hitung Document Frequency
df_freq = tfidf_obj.df(tf_matrix)
print(f"Document Frequency array shape: {df_freq.shape}")
print(f"Sample DF values (first 10): {df_freq[:10]}")

# Hitung IDF
idf_vals, idf_matrix = tfidf_obj.idf(df_freq)
print(f"\nIDF values shape: {idf_vals.shape}")
print(f"IDF diagonal matrix shape: {idf_matrix.shape}")
print(f"Sample IDF values (first 10): {idf_vals[:10]}")

### 2.3 TF-IDF Matrix (L2 Normalized)

In [ ]:
# Hitung TF-IDF (TF * IDF dengan L2 normalization per row)
tfidf_result = tfidf_obj.tfidf(tf_matrix, idf_matrix)
print(f"TF-IDF matrix shape: {tfidf_result.shape}")

# Tampilkan sebagai DataFrame
tfidf_df = pd.DataFrame(tfidf_result, columns=tf_matrix.columns)
print(f"\nSample TF-IDF (5 dokumen pertama, 10 kolom dengan nilai tertinggi):")

# Ambil 10 fitur dengan rata-rata TF-IDF tertinggi
top_features = tfidf_df.mean().nlargest(10).index
tfidf_df[top_features].head()

In [ ]:
# Visualisasi: Top 20 kata berdasarkan rata-rata TF-IDF score
mean_tfidf = tfidf_df.mean().sort_values(ascending=False)
top_20 = mean_tfidf.head(20)

plt.figure(figsize=(12, 6))
plt.barh(range(len(top_20)), top_20.values, color='steelblue')
plt.yticks(range(len(top_20)), top_20.index)
plt.xlabel('Rata-rata TF-IDF Score')
plt.title('Top 20 Kata dengan TF-IDF Tertinggi (TFIDF.py - 100 sample)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---

# Kesimpulan

1. **Vector Space Model & Cosine Distance**: Dokumen dapat direpresentasikan sebagai vektor dalam ruang dimensi tinggi. Cosine distance mengukur kesamaan antar dokumen — semakin kecil nilainya, semakin mirip kedua dokumen tersebut.

2. **TF-IDF Vectorization**: Teknik TF-IDF memberikan bobot lebih tinggi pada kata-kata yang sering muncul di suatu dokumen namun jarang muncul di dokumen lain. Ini lebih informatif dibanding raw term frequency karena mengurangi pengaruh kata-kata umum.

3. **Dr. Seuss Example**: Dengan representasi TF-IDF, kita berhasil menemukan buku Dr. Seuss yang paling mirip dengan "One Fish, Two Fish, Red Fish, Blue Fish" berdasarkan cosine distance.

4. **Tugas 1 — Klasifikasi Sentimen dengan TF-IDF (sklearn)**:
   - Dataset apps review Indonesia difilter menjadi sentimen biner (positif/negatif, tanpa netral score=3)
   - TF-IDF vectorization dengan `max_features=5000` menghasilkan representasi fitur yang compact
   - 3 classifier (Linear SVM, Logistic Regression, Multinomial NB) dilatih dan dievaluasi
   - Perbandingan menunjukkan performa masing-masing model pada dataset apps review

5. **Tugas 2 — Implementasi TFIDF.py (Wittline/tf-idf)**:
   - Class `TFIDF` dari repository Wittline berhasil dijalankan pada sample 100 review
   - Pipeline manual TF → DF → IDF → TF-IDF menghasilkan matrix TF-IDF yang ter-normalisasi (L2)
   - Visualisasi top 20 kata menunjukkan kata-kata dengan bobot TF-IDF tertinggi dalam corpus

**Referensi:**
- [Wittline/tf-idf](https://github.com/Wittline/tf-idf)
- [FarhanaTeli/Sentiment_Analysis_IMDB](https://github.com/FarhanaTeli/Sentiment_Analysis_IMDB)